openFF_lig_water.ipynb - parametrize ligand and water with openFF

input: small molecule represented as SDF or SMILES string 
make sure that moleecule's protonation state is correct

Sage forcefield is included in OpenFF

1. Generate small molecule solvated in water box. Packmol pack_box and solvate_topology create the solvated box
2. Parametrize the system with interchange
3. Export to OpenMM and run simulation
4. Minimize energy, NVT equil, NPT equil, production

In [ ]:
from openff.toolkit import Molecule, ForceField, Quantity
from openff.units import unit

In [ ]:
# smiles string representation of molecule
nadph_smiles = "NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1" # NADPH (4- at pH = 7.3). From ChEBI

# generate molecule from SMILES
nadph = Molecule.from_smiles(nadph_smiles, allow_undefined_stereo=True)
print(nadph.n_atoms, nadph.total_charge) # expect 74 atoms (C21H26N7O17P3), -4 e

# convert molecule to topology
topology = nadph.to_topology()

# Define periodicity via box vectors
topology.box_vectors = Quantity([4, 4, 4], "nanometer")

# Load OpenFF 2.3.0 "Sage"
sage = ForceField("openff-2.3.0.offxml")

# Create an Interchange object - can also use Interchange.from_smirnoff(force_field=sage, topology=topology)
out = sage.create_interchange(topology)

# Convert the Interchange object to an OpenMM System
system = out.to_openmm()

## Solvate NADPH and run MD

The cell above parametrizes NADPH alone in vacuum. Below, NADPH is packed into a water box with ions, the whole system is parametrized, and then it is minimized, equilibrated (NVT, then NPT) and run.

Solvation needs the `packmol` binary (not installed by default): `conda install -n openMM -c conda-forge packmol`

In [7]:
from collections import Counter
from openff.interchange.components._packmol import solvate_topology, RHOMBIC_DODECAHEDRON

# packmol needs 3D coordinates for the solute
nadph.generate_conformers(n_conformers=1)

# Solvate with water + 0.15 M NaCl; extra Na+ are added to neutralize NADPH's -4 charge
solvated_topology = solvate_topology(
    nadph.to_topology(),
    nacl_conc=0.15 * unit.molar,
    padding=1.2 * unit.nanometer,
    box_shape=RHOMBIC_DODECAHEDRON,
)

print(Counter(m.to_smiles(explicit_hydrogens=False) for m in solvated_topology.molecules if m.n_atoms < 5))
print("total atoms:", solvated_topology.n_atoms)
print("net charge:", sum(m.total_charge for m in solvated_topology.molecules))

Counter({'O': 1971, '[Na+]': 8, '[Cl-]': 4})
total atoms: 5999
net charge: 0.0 elementary_charge


In [ ]:
# Label residues so NADPH, water and ions can be selected in VMD/PyMOL/MDTraj (otherwise everything is "UNK")
residue_names = {"[H][O][H]": "HOH", "[Na+]": "NA", "[Cl-]": "CL"}
for i, molecule in enumerate(solvated_topology.molecules):
    resname = residue_names.get(molecule.to_smiles(), "NDP")
    for atom in molecule.atoms:
        atom.metadata["residue_name"] = resname
        atom.metadata["residue_number"] = i + 1
    if resname == "HOH":
        for atom, name in zip(molecule.atoms, ["O", "H1", "H2"]):
            atom.name = name
    elif resname in ("NA", "CL"):
        molecule.atom(0).name = resname

In [ ]:
import openmm
import openmm.app
import openmm.unit as omm_unit

# Parametrize the full system (Sage includes TIP3P water and Na+/Cl- parameters)
interchange = sage.create_interchange(solvated_topology)

# Save starting structure for visualization / as the topology file for the trajectory
interchange.to_pdb("nadph_water_start.pdb")

# Langevin dynamics at 300 K with 2 fs timestep (H-bonds and water are constrained by Sage)
integrator = openmm.LangevinMiddleIntegrator(300 * omm_unit.kelvin, 1 / omm_unit.picosecond, 2 * omm_unit.femtosecond)

# Barostat starts switched off (frequency 0) for NVT equilibration, turned on later for NPT
barostat = openmm.MonteCarloBarostat(1 * omm_unit.bar, 300 * omm_unit.kelvin, 0)

simulation = interchange.to_openmm_simulation(integrator, additional_forces=[barostat])
print("platform:", simulation.context.getPlatform().getName())

In [ ]:
# Energy minimization, then NVT equilibration: 100 ps at 300 K
simulation.minimizeEnergy()
print("minimized energy:", simulation.context.getState(getEnergy=True).getPotentialEnergy())

simulation.context.setVelocitiesToTemperature(300 * omm_unit.kelvin)
simulation.step(50_000)

In [ ]:
# NPT equilibration: 500 ps at 1 bar so the box relaxes to the right density
barostat.setFrequency(25)
simulation.context.reinitialize(preserveState=True)
simulation.step(250_000)
state = simulation.context.getState()
print("box volume after NPT:", state.getPeriodicBoxVolume())

In [ ]:
# Production MD (NPT). Saves a frame every 10 ps; adjust n_steps for length (500,000 steps x 2 fs = 1 ns)
simulation.reporters.append(openmm.app.DCDReporter("nadph_water_prod.dcd", 5000))
simulation.reporters.append(openmm.app.StateDataReporter(
    "nadph_water_prod.csv", 5000, step=True, time=True, potentialEnergy=True,
    temperature=True, volume=True, density=True, speed=True,
))
simulation.step(500_000)

# Save final state so the run can be extended later
simulation.saveState("nadph_water_prod_final.xml")

In [ ]:
import mdtraj as md
import nglview

# Load trajectory, keep NADPH centered in the box with molecules whole, and align on NADPH
traj = md.load("nadph_water_prod.dcd", top="nadph_water_start.pdb")
nadph_atoms = traj.topology.select("resname NDP")
traj.image_molecules(inplace=True, anchor_molecules=[set(traj.topology.atom(i) for i in nadph_atoms)])
traj.superpose(traj, frame=0, atom_indices=nadph_atoms)
traj.save_dcd("nadph_water_prod_centered.dcd")
print(traj)

# Show NADPH and ions (all ~2000 waters would hide NADPH; view them in VMD if needed)
view = nglview.show_mdtraj(traj)
view.clear_representations()
view.add_licorice("NDP")
view.add_spacefill("NA or CL")
view